In [0]:
%sql
WITH report_period AS (
    SELECT DATE'2026-07-01' AS month_start, last_day(DATE'2026-07-01') AS month_end
),

-- POPULATION 1: program_monthly_signups — all plans, new signups in month
signups_pop AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created'
      AND plan_id IN ('monthly1','annual1','gift-plan')
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),

-- POPULATION 2: ptd_balance — total active balance at EOM
bal_monthly_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_annual_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND plan_id = 'annual1'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_gift_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND plan_id = 'gift-plan'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_last_payment AS (
    SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
           ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'monthly1'
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
bal_last_succeeded AS (
    SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
    GROUP BY cc_user_profile_id
),
bal_monthly_dunning AS (
    SELECT DISTINCT lp.cc_user_profile_id
    FROM bal_last_payment lp
    LEFT JOIN bal_last_succeeded ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
    WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
      AND lp.reporting_datetime >= DATE_SUB((SELECT month_end FROM report_period), 15)
      AND lp.reporting_datetime <= (SELECT month_end FROM report_period)
      AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
      AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
),
a_last_payment AS (
    SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
           ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'annual1'
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
a_last_succeeded AS (
    SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
    GROUP BY cc_user_profile_id
),
a_eom_dunning AS (
    SELECT DISTINCT lp.cc_user_profile_id
    FROM a_last_payment lp
    LEFT JOIN a_last_succeeded ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
    WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
      AND lp.reporting_datetime >= DATE_SUB((SELECT month_end FROM report_period), 14)
      AND lp.reporting_datetime <= (SELECT month_end FROM report_period)
      AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
      AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
),
ptd_balance_pop AS (
    SELECT cc_user_profile_id FROM bal_monthly_active
    UNION SELECT cc_user_profile_id FROM bal_annual_active
    UNION SELECT cc_user_profile_id FROM bal_gift_active
    UNION SELECT cc_user_profile_id FROM bal_monthly_dunning
    UNION SELECT cc_user_profile_id FROM a_eom_dunning
),

-- POPULATION 3: scene_balance — P12M Scene-active CDE_IDs, minus active balance
active_balance_cde AS (
    SELECT DISTINCT s.cde_id
    FROM ptd_balance_pop p
    JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
      ON s.cc_user_profile_id = p.cc_user_profile_id
     AND s.snapshot_date_id = date_format((SELECT month_end FROM report_period), 'yyyyMMdd')
),
scene_active_cde AS (
    SELECT DISTINCT CDE_ID
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
    WHERE SCENEFLAG = 'Yes'
      AND VisitDateId BETWEEN CAST(date_format(add_months((SELECT month_end FROM report_period), -12), 'yyyyMMdd') AS INT)
                          AND CAST(date_format((SELECT month_end FROM report_period), 'yyyyMMdd') AS INT)
),
scene_balance_pop AS (
    SELECT sc.CDE_ID AS cde_id
    FROM scene_active_cde sc
    LEFT ANTI JOIN active_balance_cde ab ON CAST(sc.CDE_ID AS STRING) = ab.cde_id
),

-- DEMOGRAPHICS PIVOT (LEFT JOIN — unmatched members stay in population, fall into gender_other)
demo AS (
    SELECT 'program_monthly_signups' AS metric_type, c.gender_text, c.age_cohort_id, c.province
    FROM signups_pop p
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON c.cc_user_profile_id = p.cc_user_profile_id

    UNION ALL

    SELECT 'ptd_balance' AS metric_type, c.gender_text, c.age_cohort_id, c.province
    FROM ptd_balance_pop p
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON c.cc_user_profile_id = p.cc_user_profile_id

    UNION ALL

    SELECT 'scene_balance' AS metric_type, c.gender_text, c.age_cohort_id, c.province
    FROM scene_balance_pop p
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON CAST(c.cde_id AS STRING) = p.cde_id
)

SELECT
    (SELECT month_start FROM report_period) AS report_month,
    metric_type,

    -- Gender: unmatched/NULL falls into other
    COUNT(IF(gender_text = 'Male', 1, NULL))   AS gender_male,
    COUNT(IF(gender_text = 'Female', 1, NULL)) AS gender_female,
    COUNT(IF(gender_text NOT IN ('Male','Female') OR gender_text IS NULL, 1, NULL)) AS gender_other,

    -- Age: cohort key 1=14-18, 2=19-24, 3=25-34, 4=35-49, 5=50-64, 6=65+, 0/NULL/unmatched -> unknown
    COUNT(IF(age_cohort_id = '1', 1, NULL)) AS age_14_18,
    COUNT(IF(age_cohort_id = '2', 1, NULL)) AS age_19_24,
    COUNT(IF(age_cohort_id = '3', 1, NULL)) AS age_25_34,
    COUNT(IF(age_cohort_id = '4', 1, NULL)) AS age_35_49,
    COUNT(IF(age_cohort_id = '5', 1, NULL)) AS age_50_64,
    COUNT(IF(age_cohort_id = '6', 1, NULL)) AS age_65_plus,
    COUNT(IF(age_cohort_id = '0' OR age_cohort_id IS NULL OR age_cohort_id NOT IN ('1','2','3','4','5','6'), 1, NULL)) AS age_unknown,

    -- Province: normalized (upper/trim), ATL covers PE + PEI variants, everything else -> other
    COUNT(IF(UPPER(TRIM(province)) = 'ON', 1, NULL)) AS prov_on,
    COUNT(IF(UPPER(TRIM(province)) = 'BC', 1, NULL)) AS prov_bc,
    COUNT(IF(UPPER(TRIM(province)) = 'AB', 1, NULL)) AS prov_ab,
    COUNT(IF(UPPER(TRIM(province)) = 'QC', 1, NULL)) AS prov_qc,
    COUNT(IF(UPPER(TRIM(province)) = 'MB', 1, NULL)) AS prov_mb,
    COUNT(IF(UPPER(TRIM(province)) = 'SK', 1, NULL)) AS prov_sk,
    COUNT(IF(UPPER(TRIM(province)) IN ('NB','NS','PE','PEI','NL'), 1, NULL)) AS prov_atl,
    COUNT(IF(UPPER(TRIM(province)) NOT IN ('ON','BC','AB','QC','MB','SK','NB','NS','PE','PEI','NL')
             OR province IS NULL, 1, NULL)) AS prov_other

FROM demo
GROUP BY metric_type
ORDER BY CASE metric_type
    WHEN 'program_monthly_signups' THEN 1
    WHEN 'ptd_balance' THEN 2
    WHEN 'scene_balance' THEN 3
END